In [5]:
# module imports
from glob import glob
from IPython.display import Audio, display
import os
import json
import pandas as pd
from collections import deque
import brpylib
import numpy as np
import soundfile as sf
import subprocess
from datetime import timedelta, datetime
from joblib import Parallel, delayed
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import argparse
from scipy.signal import butter, filtfilt, iirnotch, welch, hilbert, find_peaks
from scipy.stats import kurtosis, linregress
from tqdm import tqdm
from pyNsXStitch.stitchers import StitchedNeVFile, StitchedNsXFile
from pyNsXStitch.helpers import find_nsx_in_range, get_nsx_start_timestamp, get_nsx_duration

In [6]:
# set variables
data_dir = '/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out'
patient = 'YFV'
raw_dir = '/mnt/datalake/data/emu'
save_path = f'{data_dir}/{patient}/all_convo_recording'
original_fs = 2_000
target_fs = 16_000

In [7]:
# get recording dirs
recording_dirs = glob(f'{data_dir}/{patient}/recording*')
# sort by timestamp
recording_dirs = sorted(recording_dirs, key=lambda a: datetime.strptime(os.path.basename(a).split("_")[1], "%Y%m%d-%H%M%S"))

In [9]:
original_fs = 30_000

for dir_ in tqdm(recording_dirs):
    try:
        # open metadata file
        recording_name = os.path.basename(dir_)
        metadata_path = f'{dir_}/metadata_{recording_name}.json'
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
    
        # get nevs and ns5s from rercording file
        nevs = glob(f"{raw_dir}/{patient}Datafile/DATA/{metadata['original_recording_file']}/NSP1*.nev")
        ns5s = glob(f"{raw_dir}/{patient}Datafile/DATA/{metadata['original_recording_file']}/NSP1*.ns5")
        nevs = sorted(nevs, key=lambda a: int(a[-7:-4]))
        ns5s = sorted(ns5s, key=lambda a: int(a[-7:-4]))
    
        offset = get_nsx_start_timestamp(ns5s[0])
        start_ts = int(np.round(metadata['start_stamp_s'] * original_fs)) + offset
        end_ts = int(np.round(metadata['end_stamp_s'] * original_fs)) + offset
            
        # get files to stitch
        ns5_to_stitch = find_nsx_in_range(ns5s, start_ts, end_ts)
    
        if ns5_to_stitch:
            ns5 = StitchedNsXFile(ns5_to_stitch, start=start_ts, end=end_ts, aggressive_concat=True)
            with open(f'{dir_}/neural/{patient}_{recording_name}_NSP-1.ns5', 'wb+') as f:
                ns5.write(f)
                
        # nev
        print("saving nevs")
        nev = StitchedNeVFile(nevs, start=start_ts, end=end_ts)
        with open(f'{dir_}/neural/{patient}_{recording_name}_NSP-1.nev', 'wb') as f:
            nev.write(f)
    except Exception as e:
        print(e)
        
        

  0%|          | 0/443 [00:00<?, ?it/s]

Processing new packet Timestamp: 1319453729  NPoints:2406000
Avg block write time 0:00:00.001149
saving nevs

NSP1-20260217-091906-001.nev opened

NSP1-20260217-091906-002.nev opened

NSP1-20260217-091906-003.nev opened


  0%|          | 1/443 [00:01<14:39,  1.99s/it]

[108, 108, 108]
Processing new packet Timestamp: 1324661729  NPoints:1956000
Avg block write time 0:00:00.001086


  0%|          | 2/443 [00:02<08:18,  1.13s/it]

...

100%|█████████▉| 441/443 [41:48<00:11,  5.87s/it]

Processing new packet Timestamp: 8669419925  NPoints:12047064
Processing new packet Timestamp: 8681466989  NPoints:17990925
  Packet gap: 0.0
  Concatenating packets
Processing new packet Timestamp: 8699457915  NPoints:17972336
  Packet gap: 1.0
  Concatenating packets
Avg block write time 0:00:00.004028
saving nevs

NSP1-20260220-090032-001.nev opened

NSP1-20260220-090032-002.nev opened

NSP1-20260220-090032-003.nev opened

NSP1-20260220-090032-004.nev opened

NSP1-20260220-090032-005.nev opened

NSP1-20260220-090032-006.nev opened

NSP1-20260220-090032-007.nev opened

NSP1-20260220-090032-008.nev opened

NSP1-20260220-090032-009.nev opened

NSP1-20260220-090032-010.nev opened

NSP1-20260220-090032-011.nev opened

NSP1-20260220-090032-012.nev opened

NSP1-20260220-090032-013.nev opened

NSP1-20260220-090032-014.nev opened

NSP1-20260220-090032-015.nev opened

NSP1-20260220-090032-016.nev opened

NSP1-20260220-090032-017.nev opened

NSP1-20260220-090032-018.nev opened

NSP1-20260220-0

100%|█████████▉| 442/443 [41:57<00:06,  7.00s/it]

Processing new packet Timestamp: 8717430251  NPoints:12859
Processing new packet Timestamp: 8717443109  NPoints:18005490
  Packet gap: -1.0
  Concatenating packets
Processing new packet Timestamp: 8735448599  NPoints:18015915
  Packet gap: -1.0
  Concatenating packets
Processing new packet Timestamp: 8753464515  NPoints:11976061
  Packet gap: 0.0
  Concatenating packets
Avg block write time 0:00:00.004237
saving nevs

NSP1-20260220-090032-001.nev opened

NSP1-20260220-090032-002.nev opened

NSP1-20260220-090032-003.nev opened

NSP1-20260220-090032-004.nev opened

NSP1-20260220-090032-005.nev opened

NSP1-20260220-090032-006.nev opened

NSP1-20260220-090032-007.nev opened

NSP1-20260220-090032-008.nev opened

NSP1-20260220-090032-009.nev opened

NSP1-20260220-090032-010.nev opened

NSP1-20260220-090032-011.nev opened

NSP1-20260220-090032-012.nev opened

NSP1-20260220-090032-013.nev opened

NSP1-20260220-090032-014.nev opened

NSP1-20260220-090032-015.nev opened

NSP1-20260220-090032-01

100%|██████████| 443/443 [42:07<00:00,  5.71s/it]
